# Assignment 2

**Credits**: Federico Ruggeri, Eleonora Mancini, Paolo Torroni

**Keywords**: Sexism Detection, Multi-class Classification, LLMs, Prompting


# Contact

For any doubt, question, issue or help, you can always contact us at the following email addresses:

Teaching Assistants:

* Federico Ruggeri -> federico.ruggeri6@unibo.it
* Eleonora Mancini -> e.mancini@unibo.it

Professor:

* Paolo Torroni -> p.torroni@unibo.it

# Relevant Material

- Tutorial 3
- Huggingface documentation
- Huggingface hub

In [1]:
# # PIPs
# ! pip install bitsandbytes
# ! pip install sentencepiece

In [ ]:
# Standard library
import os
import re
# import json
import gc
import random
# from collections import Counter, defaultdict

# Typing
from typing import List, Dict, Tuple, Set

# Third-party libraries
import numpy as np
import pandas as pd
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    set_seed
)
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)
from dotenv import load_dotenv
from huggingface_hub import login


DEBUG = True
if DEBUG:
    import torch
    print(torch.__version__)
    print(torch.version.cuda)

2.9.0+cu126
12.6


# Introduction

You are tasked to address the [EDOS Task B](https://github.com/rewire-online/edos) on sexism detection.

## Problem definition

Given an input text sentence, the task is to label the sentence as not-sexist or one of these four sexist categories: (1) threats, (2)  derogation, (3) animosity, (4) prejudiced discussion.

### Examples:

**Text**: *``Schedule a date with her, then don't show up. Then text her "GOTCHA B___H".''*

**Label**: Threats

**Text**: *``That’s completely ridiculous a woman flashing her boobs is not sexual assault in the slightest.''*

**Label**: Not sexist



## Approach

We will tackle the five-class classification task with LLMs.

In particular, we'll consider zero-/few-shot prompting approaches to assess the capability of some popular open-source LLMs on this task.

## Preliminaries

We are going to download LLMs from [Huggingface](https://huggingface.co/).

Many of these open-source LLMs require you to accept their "Community License Agreement" to download them.

In summary:

- If not already, create an account of Huggingface (~2 mins)
- Check a LLM model card page (e.g., [Mistral v3](https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3)) and accept its "Community License Agreement".
- Go to your account -> Settings -> Access Tokens -> Create new token -> "Repositories permissions" -> add the LLM model card you want to use.
- Save the token (we'll need it later)

### Huggingface Login

Once we have created an account and an access token, we need to login to Huggingface via code.

- Type your token and press Enter
- You can say No to Github linking

In [3]:
load_dotenv()  # Loads variables from .env
token=os.getenv("HF_TOKEN")
login(token=token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


After login, you can download all models associated with your access token in addition to those that are not protected by an access token.

### Data Loading

Since we are only interested in prompting, we do not require a train dataset.

We have preparared a small test set version of EDOS in our dedicated [Github repository](https://github.com/lt-nlp-lab-unibo/nlp-course-material).

Check the ``Assignment 2/data`` folder.
It contains:

- ``a2_test.csv`` → a small test set of 300 samples.
- ``demonstrations.csv`` -> a batch of 1000 samples for few-shot prompting.

Both datasets contain a balanced number of sexist and not sexist samples.


In [4]:
# ! rm -rf nlp-course-material
# ! git clone https://github.com/nlp-unibo/nlp-course-material.git
# # Load and encode the jsons
# data_path = "./nlp-course-material/2025-2026/Assignment 2/data/"

data_path ='./A2/data/'
a2_test = pd.read_csv(data_path + 'a2_test.csv')
demostration = pd.read_csv(data_path + 'demonstrations.csv')

# add column with label index category
label2idx = {
            'not-sexist': 0,
            'threats': 1,
            'derogation': 2,
            'animosity': 3,
            'prejudiced': 4
            }
a2_test['label_index'] = a2_test['label_category'].map(label2idx)
demostration['label_index'] = demostration['label_category'].map(label2idx)

### Instructions

We require you to:

* **Download** the ``A2/data`` folder.
* **Encode** ``a2_test.csv`` into a ``pandas.DataFrame`` object.

# [Task 1 - 0.5 points] Model setup

Once the test data has been loaded, we have to setup the model pipeline for inference.

In particular, we have to:
- Load the model weights from Huggingface
- Quantize the model to fit into a single-GPU limited hardware

## Which LLMs?

The pool of LLMs is ever increasing and it's impossible to keep track of all new entries.

We focus on popular open-source models.

- [Mistral v2](https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2)
- [Mistral v3](https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3)
- [Llama v3.1](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct)
- [Phi3-mini](https://huggingface.co/microsoft/Phi-3-mini-4k-instruct)
- [TinyLlama](https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0)
- [DeepSeek-R1](https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Qwen-7B)
- [Qwen3](https://huggingface.co/Qwen/Qwen3-1.7B)

Other open-source models are more than welcome!

### Instructions

In order to get Task 1 points, we require you to:

* Pick 2 model cards from the provided list.
* For each model:
  - Setup a quantization configuration for the model.
  - Load the model via HuggingFace APIs.


### Note

There's a popular library integrated with Huggingface's ``transformers`` to perform quantization.


In [5]:
def setup_model(card_name, local_path="myDrive/models/"):

    model_path = local_path + card_name.replace("/", "_")

    if not os.path.exists(local_path) : os.makedirs(local_path)
    if not os.path.exists(model_path):
        tokenizer = AutoTokenizer.from_pretrained(card_name)
        tokenizer.pad_token = tokenizer.eos_token

        model_bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16
        )

        model = AutoModelForCausalLM.from_pretrained(
            card_name,
            return_dict=True,
            quantization_config=model_bnb_config,
            device_map='auto',
            dtype=torch.bfloat16
        )

        model.save_pretrained(model_path)
        tokenizer.save_pretrained(model_path)

    else:
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        tokenizer.pad_token = tokenizer.eos_token

        model = AutoModelForCausalLM.from_pretrained(
            model_path,
            return_dict=True,
            device_map='auto',
            dtype=torch.bfloat16
        )

    model_gen_config = model.generation_config
    model_gen_config.max_new_tokens = 5000 if card_name == "Qwen/Qwen3-1.7B" else 500
    model_gen_config.eos_token_id = tokenizer.eos_token_id
    model_gen_config.pad_token_id = tokenizer.eos_token_id
    model_gen_config.temperature = None
    model_gen_config.num_return_sequences = 1

    tokenizer.padding_side = 'left'  # for batch generation

    return model, tokenizer

In [6]:
# import torch
# import gc

# # ...existing code...
# def clear_memory():
#     gc.collect()
#     torch.cuda.empty_cache()

# del tiny_llama_model
# clear_memory()
# del mistral_model
# clear_memory()
# del quen_model
# clear_memory()
# del deepseek_model
# clear_memory()

In [7]:
mistral_card = "mistralai/Mistral-7B-Instruct-v0.3"
# deepseek_card = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
# tiny_llama_card = "TinyLlama/TinyLlama-1.1B-Chat-v1.0" 
quen_card = "Qwen/Qwen3-1.7B"

# print("Loading Tiny-Llama...")
# tiny_llama_model, tiny_llama_tokenizer = setup_model(tiny_llama_card)
print("Loading Qwen-3...")
quen_model, quen_tokenizer = setup_model(quen_card)
print("Loading Mistral-3...")
mistral_model, mistral_tokenizer = setup_model(mistral_card)
# print("Loading DeepSeek-R1")
# deepseek_model, deepseek_tokenizer = setup_model(deepseek_card)

models = {
    # "tiny-llama": {
    #     "card": tiny_llama_card,
    #     "model": tiny_llama_model,
    #     "tokenizer": tiny_llama_tokenizer
    # },
    "quen-3": {
        "card": quen_card,
        "model": quen_model,
        "tokenizer": quen_tokenizer
    },
    "mistral": {
        "card": mistral_card,
        "model": mistral_model,
        "tokenizer": mistral_tokenizer
    },
    # "deepseek": {
    #     "card": deepseek_card,
    #     "model": deepseek_model,
    #     "tokenizer": deepseek_tokenizer
    # }
}


Loading Qwen-3...


The tokenizer you are loading from 'myDrive/models/Qwen_Qwen3-1.7B' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Loading Mistral-3...


# [Task 2 - 1.0 points] Prompt setup

Prompting requires an input pre-processing phase where we convert each input example into a specific instruction prompt.


## Prompt Template

Use the following prompt template to process input texts.

In [8]:
prompt = [
    {
        'role': 'system',
        'content': 'You are an annotator for sexism detection.'
    },
    {
        'role': 'user',
        'content': """Your task is to classify input text as not-sexist 
         or sexist. If sexist, classify input text according to one
         of the following four categories: threats, derogation,
         animosity, prejudiced discussion.
         
         Below you find sexist categories definitions:
         Threats: the text expresses intent or desire to harm a woman.
         Derogation: the text describes a woman in a derogative manner.
         Animosity: the text contains slurs or insults towards a woman.
         Prejudiced discussion: the text expresses supports for
         mistreatment of women as individuals.
        
         Respond only by writing one of the following categories:
         not-sexist, threats, derogation, animosity, prejudiced.

        TEXT: {text}

        ANSWER:
        """
    }
]

In [9]:
if DEBUG:
    for name, model in models.items():
        print(f"Model: {name}")
        formatted_prompt = model["tokenizer"].apply_chat_template(
            prompt,
            tokenize=False,
            add_generation_prompt=True
        )
        print(formatted_prompt)
        print("*"*50)

Model: quen-3
<|im_start|>system
You are an annotator for sexism detection.<|im_end|>
<|im_start|>user
Your task is to classify input text as not-sexist 
         or sexist. If sexist, classify input text according to one
         of the following four categories: threats, derogation,
         animosity, prejudiced discussion.

         Below you find sexist categories definitions:
         Threats: the text expresses intent or desire to harm a woman.
         Derogation: the text describes a woman in a derogative manner.
         Animosity: the text contains slurs or insults towards a woman.
         Prejudiced discussion: the text expresses supports for
         mistreatment of women as individuals.

         Respond only by writing one of the following categories:
         not-sexist, threats, derogation, animosity, prejudiced.

        TEXT: {text}

        ANSWER:
        <|im_end|>
<|im_start|>assistant

**************************************************
Model: mistral
<s>[INST] 

### Instructions

In order to get Task 2 points, we require you to:

* Write a ``prepare_prompts`` function as the one reported below.

In [ ]:
# def prepare_prompts(texts, prompt_template, tokenizer, demonstrations_str=""):
#     """
#         This function format input text samples into instructions prompts.

#         Inputs:
#         texts: input texts to classify via prompting
#         prompt_template: the prompt template provided in this assignment
#         tokenizer: the transformers Tokenizer object instance associated
#         with the chosen model card

#         Outputs:
#         input texts to classify in the form of instruction prompts
#     """
#     # Apply chat template
#     prompt_template_chat = tokenizer.apply_chat_template(
#         prompt_template, tokenize=False, add_generation_prompt=True
#     )
    
#     # Format prompts with demonstrations and text
#     formatted_prompts_chat = [
#         prompt_template_chat.format(examples=demonstrations_str, text=t) 
#         for t in texts
#     ]
    
#     # Tokenize
#     tokenized_prompts_chat = tokenizer(
#         formatted_prompts_chat, 
#         padding=True, 
#         truncation=True,
#         return_tensors='pt'
#     ).to('cuda')
    
#     return tokenized_prompts_chat

In [11]:
def prepare_prompts(texts, prompt_template, tokenizer):
  """
    This function format input text samples into instructions prompts.

    Inputs:
      texts: input texts to classify via prompting
      prompt_template: the prompt template provided in this assignment
      tokenizer: the transformers Tokenizer object instance associated
      with the chosen model card

    Outputs:
      input texts to classify in the form of instruction prompts
  """
  # Make the single message as chat
  prompt_template_chat = tokenizer.apply_chat_template(
      prompt_template, tokenize=False, add_generation_prompt=True
  )
  # Substitue the text placeholder with the text
  formatted_prompts_chat = [prompt_template_chat.format(text = t) for t in texts]

  # Tokenize the formatted chat for the model
  tokenized_prompts_chat = tokenizer(
      formatted_prompts_chat, padding=True, truncation=True,
      return_tensors='pt'
  ).to('cuda')

  return tokenized_prompts_chat


### Notes

1. You are free to modify the prompt format (**not its content**) as you like depending on your code implementation.

2. Note that the provided prompt has placeholders. You need to format the string to replace placeholders. Huggingface might have dedicated APIs for this.

# [Task 3 - 1.0 points] Inference

We are now ready to define the inference loop where we prompt the model with each pre-processed sample.

### Instructions

In order to get Task 3 points, we require you to:

* Write a ``generate_responses`` function as the one reported below.
* Write a ``process_response`` function as the one reported below.

In [12]:
#   # gpu version slicing
#   generated_ids_list = []
#   for i in range(response_ids.size(0)):
#       input_len = prompt_examples['input_ids'][i].size(0)
#       generated_ids = response_ids[i][input_len:]
#       generated_ids_list.append(generated_ids)
#   input_lens = response_ids[:]
#   generated_ids_list = [generated_ids_only[i][lengths[i]:] for i in range(batch_size)]
# ----------------------------

# def predict(model, tokenizer, texts_df, prompt_template, demonstrations="", verbose=False, batch_size=None):
#     """
#     This function implements the full prediction pipeline
#     for a given LLM model.

#     Inputs:
#       model: LLM model instance for prompting
#       tokenizer: the transformers Tokenizer object instance
#       texts_df: input texts DataFrame to classify via prompting
#       prompt_template: the prompt template provided in this assignment
#       demonstrations: optional demonstration string for few-shot
#       verbose: whether to print debug information
#       batch_size: how many samples to take from texts_df (None = all)
      
#     Outputs:
#       parsed LLM responses (numpy array of predicted labels)
#     """
#     # Determine how many samples to process
#     if batch_size is None:
#         batch_size = len(texts_df)
    
#     texts = texts_df['text'].tolist()[:batch_size]
#     truth_labels = texts_df['label'].values[:batch_size]
    
#     # Prepare prompts
#     prompt_examples = prepare_prompts(
#         texts,
#         prompt_template,
#         tokenizer,
#         demonstrations_str=demonstrations
#     )
    
#     # Generate responses
#     response_ids = generate_responses(model, prompt_examples)
    
#     # Extract only generated tokens (skip input prompt)
#     decoded_outputs = []
#     for i, response in enumerate(response_ids):
#         # Get input length for this sample
#         input_len = prompt_examples['input_ids'][i].shape[0]
        
#         # Extract only generated tokens
#         generated_ids = response[input_len:]
        
#         # Decode
#         generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
#         decoded_outputs.append(generated_text)
    
#     # Process responses to get labels
#     predictions = np.array([process_response(output) for output in decoded_outputs])
    
#     if verbose:
#         print("-"*50)
#         for i, output in enumerate(decoded_outputs):
#             print(f"Input text: {texts[i]}")
#             print(f"Generated output: {output}")
#             print(f"Predicted label: {predictions[i]}")
#             print(f"Truth label: {truth_labels[i]}")
#             print("-"*30)
    
#     return predictions

In [13]:
def generate_responses(model, prompt_examples, max_new_tokens=10):
  """
    This function implements the inference loop for a LLM model.
    Given a set of examples, the model is tasked to generate
    a response.

    Inputs:
      model: LLM model instance for prompting
      prompt_examples: pre-processed text samples

    Outputs:
      generated responses
  """

  # Initial generation forcing max_new_tokens=10
  response_ids = model.generate(
      input_ids=prompt_examples['input_ids'],
      attention_mask=prompt_examples['attention_mask'],
      do_sample=False,
      max_new_tokens=max_new_tokens
    )

  return response_ids

In [14]:
def process_response(response):
  """
    This function takes a textual response generated by the LLM
    and processes it to map the response to a binary label.

    Inputs:
      response: generated response from LLM

    Outputs:
      parsed classification response.
      Use the following mapping:
      {
        'not-sexist': 0,
        'threats': 1,
        'derogation': 2,
        'animosity': 3,
        'prejudiced': 4
      }
  """

  mapping = {
    'not-sexist': 0,
    'threats': 1,
    'derogation': 2,
    'animosity': 3,
    'prejudiced': 4
  }

  # Normalize text
  text = response.lower().strip()
  
  # Try exact matches first
  for key, value in mapping.items():
      if key in text:
          return value
  
  # Try partial matches (in case of slight variations)
  if 'threat' in text:
      return 1
  elif 'derogat' in text:
      return 2
  elif 'animosity' in text or 'animosi' in text:
      return 3
  elif 'prejudice' in text:
      return 4
  elif 'not' in text and 'sexist' in text:
      return 0
  
  # Default: failed response, assign to not-sexist
  return 0


In [15]:
def process_response_count(response):
    """
    This function takes a textual response generated by the LLM
    and processes it to map the response to a binary label.

    Inputs:
      response: generated response from LLM

    Outputs:
      parsed classification response.
      Use the following mapping:
      {
        'not-sexist': 0,
        'threats': 1,
        'derogation': 2,
        'animosity': 3,
        'prejudiced': 4
      }
    """

    mapping = {
        'not-sexist': 0,
        'threats': 1,
        'derogation': 2,
        'animosity': 3,
        'prejudiced': 4
    }

    # Normalize text
    text = response.lower().strip()
    count_mapping = {
        0: 0,
        1: 0,
        2: 0,
        3: 0,
        4: 0
    }

    # Try exact matches first
    for key, value in mapping.items():
        if key in text:
            count_mapping[value] += 1

    # Try partial matches (in case of slight variations)
    if 'threat' in text:
        count_mapping[1] += 1
    elif 'derogat' in text:
        count_mapping[2] += 1
    elif 'animosity' in text or 'animosi' in text:
        count_mapping[3] += 1
    elif 'prejudice' in text:
        count_mapping[4] += 1
    elif 'not' in text and 'sexist' in text:
        count_mapping[0] += 1

    # return the label with the highest count
    return np.argmax(count_mapping, key=count_mapping.get)


## Notes

1. According to our tests, it should take you ~10 mins to perform full inference on 300 samples on Colab.

In [16]:
map_think_quen = {"<think>": 151667, "</think>": 151668}

In [20]:
def clean_and_process(text):
    result = None
    if "<think>" in text and "</think>" not in text:
        # result = process_response_count(text)
        print("\t - Incomplete <think> tag -> inferred")
    else : #"</think>" in text
        text = text.split("</think>")[-1].strip()
        result = process_response(text)
    return result


def predict(model, tokenizer, texts_df, prompt_template, demonstrations="", verbose = False, batch_size=None):
  """
    This function implements the full prediction pipeline
    for a given LLM model.

    Inputs:
      model: LLM model instance for prompting
      tokenizer: the transformers Tokenizer object instance
      texts_df: input texts DataFrame to classify via prompting
      prompt_template: the prompt template provided in this assignment
    Outputs:
      parsed LLM responses
  """
  if batch_size is None:
      batch_size = len(texts_df)

  if verbose:
      print(f"Predicting on {batch_size} samples...")

  texts = texts_df['text'].tolist()[:batch_size]
  truth_labels = texts_df['label_category'].values[:batch_size]

  if verbose:
      print("Preparing prompts...")
  
  # Prepare prompts
  prompt_examples = prepare_prompts(
      texts,
      prompt_template,
      tokenizer,
    #   demonstrations_str=demonstrations
  )

  if verbose:
      print("Generating responses...")

  # Generate responses
  response_ids = generate_responses(
      model,
      prompt_examples,
      max_new_tokens=100
  )

  if verbose:
      print("Cleaning generated outputs...")
  
  prompt_len = prompt_examples['input_ids'].shape[1]
  generated_only = response_ids[:, prompt_len:]

  if verbose:
      print("start decoding...")
  decoded_outputs = tokenizer.batch_decode(
      generated_only, 
      skip_special_tokens=True
  )

  # Delete thinking in the output if there is any
  predictions = np.array([clean_and_process(t) for t in decoded_outputs])
  
  if verbose:
      for i, output in enumerate(decoded_outputs):
        print("-"*50)
        print(f"Input text: {texts[i]}")
        print(f"Generated output: {output}")
        print(f"Predicted label: {predictions[i]}")
        print(f"Truth label: {truth_labels[i]}")
  return predictions


In [21]:
predictions = {}
for name, model_info in models.items():
    print("-*"*30)
    print(f"Model: {name}")
    print("-*"*30)
    predictions[name] = predict(
        model_info['model'], 
        model_info['tokenizer'], 
        a2_test, prompt, 
        verbose = True, 
        batch_size=20
        )

-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*
Model: quen-3
-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*
Predicting on 20 samples...
Preparing prompts...
Generating responses...
Cleaning generated outputs...
start decoding...
	 - Incomplete <think> tag -> inferred
	 - Incomplete <think> tag -> inferred
	 - Incomplete <think> tag -> inferred
	 - Incomplete <think> tag -> inferred
	 - Incomplete <think> tag -> inferred
	 - Incomplete <think> tag -> inferred
	 - Incomplete <think> tag -> inferred
	 - Incomplete <think> tag -> inferred
	 - Incomplete <think> tag -> inferred
	 - Incomplete <think> tag -> inferred
	 - Incomplete <think> tag -> inferred
	 - Incomplete <think> tag -> inferred
	 - Incomplete <think> tag -> inferred
	 - Incomplete <think> tag -> inferred
	 - Incomplete <think> tag -> inferred
	 - Incomplete <think> tag -> inferred
	 - Incomplete <think> tag -> inferred
	 - Incomplete <think> tag -> inferred
	 - Incomplete <think> tag -> inferred
	 

In [ ]:

if DEBUG:
    for name, model_info in models.items():
        print("-*"*30)
        print(f"Model: {name}")
        print("-*"*30)
        sample_texts = a2_test['text'].tolist()[:15]
        truth_labels = a2_test['label_category'].values[:15]
        tokenized_prompts = prepare_prompts(
            sample_texts,
            prompt,
            model_info["tokenizer"]
        )
        responses = generate_responses(
            model_info["model"],
            tokenized_prompts
        )
        
        # Extract only generated tokens (skip input)
        for i, response in enumerate(responses):
            input_len = tokenized_prompts['input_ids'][i].shape[0]
            generated_ids = response[input_len:]  # Skip input tokens
            generated_text = model_info["tokenizer"].decode(
                generated_ids, 
                skip_special_tokens=True
            )
            if "</think>" in generated_text:
                generated_text = generated_text.split("</think>")[-1].strip()
            
            label = process_response(generated_text)
            print(f"Text: {sample_texts[i]}")
            print(f"Generated: {generated_text}")
            print(f"Label: {label}")
            print(f"Truth label: {truth_labels[i]}")
            print("-"*30)

# [Task 4 - 0.5 points] Metrics

In order to evaluate selected LLMs, we need to compute performance metrics.

We compute **macro F1-score** and the ratio of failed responses generated by models (**fail-ratio**). 

That is, how frequent the LLM fails to follow instructions and provides incorrect responses that do not address the classification task.

In summary, we parse generated responses as follows:
- **0** if 'not-sexist'
- **1** if 'threats'
- **2** if 'derogation'
- **3** if 'animosity'
- **4** if 'prejudiced'
- **0** if the model does not answer in either way

### Instructions

In order to get Task 4 points, we require you to:

* Write a ``compute_metrics`` function as the one reported below.
* Compute metrics for the two selected LLMs.

In [ ]:
def compute_metrics(y_pred, y_true):
  """
    This function takes predicted and ground-truth labels and compute
    metrics. In particular, this function compute accuracy and
    fail-ratio metrics. This function internally invokes
    `process_response` to compute metrics.

    Inputs:
      y_pred: parsed LLM responses
      y_true: ground-truth labels

    Outputs:
      dictionary containing desired metrics
  """
  # Calculate metrics
  accuracy = accuracy_score(y_true, y_pred)
  macro_f1 = f1_score(y_true, y_pred, average='macro')
  macro_precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
  macro_recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
  
  # TODO: Calculate per-class F1
  # per_class_f1 = f1_score(y_true, y_pred, average=None, zero_division=0)
  
  return {
      'accuracy': accuracy,
      'macro_f1': macro_f1,
      'macro_precision': macro_precision,
      'macro_recall': macro_recall,
      # 'per_class_f1': per_class_f1
  }

In [ ]:
predictions = predict(
    models['quen-3']['model'],
    models['quen-3']['tokenizer'],
    a2_test,
    prompt
)



In [ ]:
computed_metrics = compute_metrics(
    predictions,
    a2_test['label_index'].values
)

In [ ]:
pd.DataFrame([computed_metrics])

# [Task 5 - 1.0 points] Few-shot Inference

So far, we have tested models in a zero-shot fashion: we provide the input text to classify and instruct the model to generate a response.

We are now interested in performing few-shot prompting to see the impact of providing demonstration examples.

To do so, we slightly change the prompt template as follows.

In [ ]:
prompt = [
    {
        'role': 'system',
        'content': 'You are an annotator for sexism detection.'
    },
    {
        'role': 'user',
        'content': """Your task is to classify input text as not-sexist 
         or sexist. If sexist, classify input text according to one
         of the following four categories: threats, derogation,
         animosity, prejudiced discussion.
         
         Below you find sexist categories definitions:
         Threats: the text expresses intent or desire to harm a woman.
         Derogation: the text describes a woman in a derogative manner.
         Animosity: the text contains slurs or insults towards a woman.
         Prejudiced discussion: the text expresses supports for
         mistreatment of women as individuals.
        
         Respond only by writing one of the following categories:
         not-sexist, threats, derogation, animosity, prejudiced.

        EXAMPLES: {examples}

        TEXT: {text}

        ANSWER:
        """
    }
]

The new prompt template reports some demonstration examples to instruct the model.

Generally, we provide an equal number of demonstrations per class as shown in the example below.

In [ ]:
# prompt = [
#     {
#         'role': 'system',
#         'content': 'You are an annotator for sexism detection.'
#     },
#     {
#         'role': 'user',
#         'content': """Your task is to classify input text as not-sexist 
#          or sexist. If sexist, classify input text according to one
#          of the following four categories: threats, derogation,
#          animosity, prejudiced discussion.
         
#          Below you find sexist categories definitions:
#          Threats: the text expresses intent or desire to harm a woman.
#          Derogation: the text describes a woman in a derogative manner.
#          Animosity: the text contains slurs or insults towards a woman.
#          Prejudiced discussion: the text expresses supports for
#          mistreatment of women as individuals.
        
#          Respond only by writing one of the following categories:
#          not-sexist, threats, derogation, animosity, prejudiced.
         
#          EXAMPLES:
#          TEXT: **example 1**
#          ANSWER: threats
#          TEXT: **example 2**
#          ANSWER: not-sexist

#          TEXT: {text}

#         ANSWER:
#         """
#     }
# ]

## Instructions

In order to get Task 5 points, we require you to:

- Load ``demonstrations.csv`` and encode it into a ``pandas.DataFrame`` object.
- Define a ``build_few_shot_demonstrations`` function as the one reported below.
- Modify ``prepare_prompts`` to support demonstrations.
- Perform few-shot inference as in Task 3.
- Compute metrics as in Task 4.

In [ ]:
def build_few_shot_demonstrations(demonstrations_df, num_per_class=1):
    """
    Build few-shot demonstrations with balanced examples per class.
    
    Inputs:
      demonstrations_df: DataFrame wrapping demonstrations.csv
      num_per_class: number of demonstrations per class
    
    Outputs:
      formatted string to inject into prompt template
    """
    label_mapping = {
        0: 'not-sexist',
        1: 'threats',
        2: 'derogation',
        3: 'animosity',
        4: 'prejudiced'
    }
    
    examples_str = ""
    
    # Sample from each class
    for label_id in range(5):
        class_samples = demonstrations_df[demonstrations_df['label_category'] == label_id]
        sampled = class_samples.sample(
            n=min(num_per_class, len(class_samples)), 
            random_state=42
        )
        
        for _, row in sampled.iterrows():
            examples_str += f"TEXT: {row['text']}\nANSWER: {label_mapping[label_id]}\n"
    
    return examples_str


In [ ]:
# Few-shot (with demonstrations)
demo_str = build_few_shot_demonstrations(demostration, num_per_class=1)
predictions = predict(
    models['quen-3']['model'],
    models['quen-3']['tokenizer'],
    a2_test['text'].tolist()[:3],
    prompt,
    demonstrations=demo_str
)
computed_metrics = compute_metrics(
    predictions,
    a2_test['label_category'].values[:3]
)
pd.DataFrame([computed_metrics])

## Notes

1. You are free to pick any value for ``num_per_class``.

2. According to our tests, few-shot prompting increases inference time by some minutes (we experimented with ``num_per_class`` $\in [2, 4]$).

# [Task 6 - 1.0 points] Error Analysis

We are now interested in evaluating model responses and comparing their performance.

This analysis helps us in understanding

- Classification task performance gap: are the models good at this task?
- Generation quality: which kind of responses do models generate?
- Errors: which kind of mistakes do models do?

### Instructions

In order to get Task 6 points, we require you to:

* Compare classification performance of selected LLMs in a Table.
* Compute confusion matrices for selected LLMs.
* Briefly summarize your observations on generated responses.

# [Task 7 - 1.0 points] Report

Wrap up your experiment in a short report (up to 2 pages).

### Instructions

* Use the NLP course report template.
* Summarize each task in the report following the provided template.

### Recommendations

The report is not a copy-paste of graphs, tables, and command outputs.

* Summarize classification performance in Table format.
* **Do not** report command outputs or screenshots.
* Report learning curves in Figure format.
* The error analysis section should summarize your findings.

# Submission

* **Submit** your report in PDF format.
* **Submit** your python notebook.
* Make sure your notebook is **well organized**, with no temporary code, commented sections, tests, etc...

# FAQ

Please check this frequently asked questions before contacting us.

### Model cards

You can pick any open-source model card you like.

We recommend starting from those reported in this assignment.

### Implementation

Everything can be done via ``transformers`` APIs.

However, you are free to test frameworks, such as [LangChain](https://www.langchain.com/), [LlamaIndex](https://www.llamaindex.ai/) [LitParrot](https://github.com/awesome-software/lit-parrot), provided that you correctly address task instructions.

### Task Performance

The task is challenging and zero-shot prompting may show relatively low performance depending on the chosen model.

### Prompt Template

Do not change the provided prompt template.

You are only allowed to change it in case of a possible extension.

### Optimizations

Any kind of code optimization (e.g., speedup model inference or reduce computational cost) is more than welcome!

### Bonus Points

0.5 bonus points are arbitrarily assigned based on significant contributions such as:

- Outstanding error analysis
- Masterclass code organization
- Evaluate A1 dataset and perform comparison
- Perform prompt tuning

Note that bonus points are only assigned if all task points are attributed (i.e., 6/6).

# The End

In [ ]:



# ========== Task 5: Few-Shot Demonstrations ==========
def build_few_shot_demonstrations(demonstrations_df, num_per_class=2):
    """
    Build balanced few-shot demonstration examples.
    
    Inputs:
        demonstrations_df: DataFrame wrapping demonstrations.csv
        num_per_class: number of demonstrations per class
    
    Outputs:
        list of demonstration dictionaries
    """
    label_mapping = {
        0: 'not-sexist',
        1: 'threats',
        2: 'derogation',
        3: 'animosity',
        4: 'prejudiced'
    }
    
    demonstrations = []
    
    # Sample from each class
    for label_id in range(5):
        class_samples = demonstrations_df[demonstrations_df['label'] == label_id]
        sampled = class_samples.sample(n=min(num_per_class, len(class_samples)), random_state=42)
        
        for _, row in sampled.iterrows():
            demonstrations.append({
                'text': row['text'],
                'label': label_mapping[label_id]
            })
    
    return demonstrations


# ========== Complete Inference Pipeline ==========
def run_inference(model, tokenizer, test_df, prompt_template, demonstrations=None, batch_size=8):
    """
    Run complete inference pipeline on test set.
    
    Inputs:
        model: LLM model
        tokenizer: model tokenizer
        test_df: test DataFrame
        prompt_template: prompt template
        demonstrations: optional few-shot examples
        batch_size: batch size for inference
    
    Outputs:
        list of predicted labels
    """
    predictions = []
    texts = test_df['text'].tolist()
    
    # Process in batches
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        
        # Prepare prompts
        tokenized_inputs = prepare_prompts(
            batch_texts, 
            prompt_template, 
            tokenizer, 
            demonstrations
        )
        
        # Generate responses
        with torch.no_grad():
            outputs = generate_responses(model, tokenized_inputs)
        
        # Decode and process responses
        for j, output in enumerate(outputs):
            # Skip the input tokens
            input_len = tokenized_inputs['input_ids'][j].shape[0]
            response_ids = output[input_len:]
            response_text = tokenizer.decode(response_ids, skip_special_tokens=True)
            
            # Process response
            pred_label = process_response(response_text)
            predictions.append(pred_label)
            
            if DEBUG and i == 0 and j < 2:
                print(f"Text: {batch_texts[j][:100]}...")
                print(f"Response: {response_text[:200]}")
                print(f"Predicted: {pred_label}\n")
    
    return predictions


# ========== Evaluation and Visualization ==========
def evaluate_and_visualize(y_pred, y_true, model_name):
    """
    Evaluate predictions and create visualizations.
    """
    # Compute metrics
    metrics = compute_metrics(y_pred, y_true)
    
    # Print metrics
    print(f"\n{'='*50}")
    print(f"Results for {model_name}")
    print(f"{'='*50}")
    print(f"Accuracy: {metrics['accuracy']:.4f}")
    print(f"Macro F1: {metrics['macro_f1']:.4f}")
    print(f"Macro Precision: {metrics['macro_precision']:.4f}")
    print(f"Macro Recall: {metrics['macro_recall']:.4f}")
    
    print("\nPer-class F1 scores:")
    class_names = ['not-sexist', 'threats', 'derogation', 'animosity', 'prejudiced']
    for i, (name, f1) in enumerate(zip(class_names, metrics['per_class_f1'])):
        print(f"  {name}: {f1:.4f}")
    
    # Classification report
    print("\n" + classification_report(y_true, y_pred, target_names=class_names, zero_division=0))
    
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    
    # Plot confusion matrix
    plt.figure(figsize=(10, 8))
    plt.imshow(cm, interpolation='nearest', cmap='Blues')
    plt.title(f'Confusion Matrix - {model_name}')
    plt.colorbar()
    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=45, ha='right')
    plt.yticks(tick_marks, class_names)
    
    # Add text annotations
    thresh = cm.max() / 2.
    for i, j in np.ndindex(cm.shape):
        plt.text(j, i, format(cm[i, j], 'd'),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black")
    
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()
    plt.savefig(f'confusion_matrix_{model_name.replace(" ", "_")}.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    return metrics


# ========== Main Execution ==========
# Load demonstration data for few-shot
demonstrations_df = pd.read_csv(data_path + 'demonstrations.csv')

# Build few-shot demonstrations (2 per class)
few_shot_demos = build_few_shot_demonstrations(demonstrations_df, num_per_class=2)

# Define prompt templates
zero_shot_prompt = [
    {'role': 'system', 'content': 'You are an annotator for sexism detection.'},
    {'role': 'user', 'content': """Your task is to classify input text as not-sexist 
     or sexist. If sexist, classify input text according to one
     of the following four categories: threats, derogation,
     animosity, prejudiced discussion.
     
     Below you find sexist categories definitions:
     Threats: the text expresses intent or desire to harm a woman.
     Derogation: the text describes a woman in a derogative manner.
     Animosity: the text contains slurs or insults towards a woman.
     Prejudiced discussion: the text expresses supports for
     mistreatment of women as individuals.
    
     Respond only by writing one of the following categories:
     not-sexist, threats, derogation, animosity, prejudiced.

    TEXT: {text}

    ANSWER:
    """}
]

few_shot_prompt = [
    {'role': 'system', 'content': 'You are an annotator for sexism detection.'},
    {'role': 'user', 'content': """Your task is to classify input text as not-sexist 
     or sexist. If sexist, classify input text according to one
     of the following four categories: threats, derogation,
     animosity, prejudiced discussion.
     
     Below you find sexist categories definitions:
     Threats: the text expresses intent or desire to harm a woman.
     Derogation: the text describes a woman in a derogative manner.
     Animosity: the text contains slurs or insults towards a woman.
     Prejudiced discussion: the text expresses supports for
     mistreatment of women as individuals.
    
     Respond only by writing one of the following categories:
     not-sexist, threats, derogation, animosity, prejudiced.

    EXAMPLES: {examples}

    TEXT: {text}

    ANSWER:
    """}
]

# Store results
results = {}

# Evaluate each model with zero-shot and few-shot
for model_name, model_dict in models.items():
    print(f"\n{'#'*60}")
    print(f"Evaluating {model_name}")
    print(f"{'#'*60}")
    
    model = model_dict['model']
    tokenizer = model_dict['tokenizer']
    
    # Zero-shot evaluation
    print(f"\n--- Zero-shot Prompting ---")
    zero_shot_preds = run_inference(
        model, tokenizer, a2_test, zero_shot_prompt, 
        demonstrations=None, batch_size=8
    )
    zero_shot_metrics = evaluate_and_visualize(
        zero_shot_preds, a2_test['label'].tolist(), 
        f"{model_name} (Zero-shot)"
    )
    
    # Few-shot evaluation
    print(f"\n--- Few-shot Prompting ---")
    few_shot_preds = run_inference(
        model, tokenizer, a2_test, few_shot_prompt,
        demonstrations=few_shot_demos, batch_size=4  # Smaller batch for few-shot
    )
    few_shot_metrics = evaluate_and_visualize(
        few_shot_preds, a2_test['label'].tolist(),
        f"{model_name} (Few-shot)"
    )
    
    # Store results
    results[model_name] = {
        'zero_shot': zero_shot_metrics,
        'few_shot': few_shot_metrics
    }

# ========== Summary Table ==========
print("\n" + "="*80)
print("SUMMARY TABLE")
print("="*80)
print(f"{'Model':<25} {'Setting':<15} {'Accuracy':<12} {'Macro F1':<12}")
print("-"*80)
for model_name, res in results.items():
    for setting, metrics in res.items():
        setting_name = "Zero-shot" if setting == "zero_shot" else "Few-shot"
        print(f"{model_name:<25} {setting_name:<15} {metrics['accuracy']:<12.4f} {metrics['macro_f1']:<12.4f}")
print("="*80)